In [4]:
from pathlib import Path
import math

import pandas as pd
from IPython.display import display

GRID_CSV_PATH = Path("tainan_hex.csv")


def twd97_to_lonlat(x: float, y: float) -> tuple[float, float]:
    """Convert EPSG:3826 (TWD97 / TM2 zone 121) x/y to lon/lat."""
    a = 6378137.0
    b = 6356752.314245
    lon0 = math.radians(121.0)
    k0 = 0.9999
    dx = 250000.0

    e = math.sqrt(1 - (b * b) / (a * a))
    e2 = e * e
    e1 = (1 - math.sqrt(1 - e2)) / (1 + math.sqrt(1 - e2))
    x = float(x) - dx
    y = float(y)

    m = y / k0
    mu = m / (a * (1 - e2 / 4 - 3 * e2**2 / 64 - 5 * e2**3 / 256))
    j1 = 3 * e1 / 2 - 27 * e1**3 / 32
    j2 = 21 * e1**2 / 16 - 55 * e1**4 / 32
    j3 = 151 * e1**3 / 96
    j4 = 1097 * e1**4 / 512
    fp = mu + j1 * math.sin(2 * mu) + j2 * math.sin(4 * mu) + j3 * math.sin(6 * mu) + j4 * math.sin(8 * mu)

    ep2 = e2 / (1 - e2)
    sin_fp = math.sin(fp)
    cos_fp = math.cos(fp)
    tan_fp = math.tan(fp)
    c1 = ep2 * cos_fp**2
    t1 = tan_fp**2
    r1 = a * (1 - e2) / (1 - e2 * sin_fp**2) ** 1.5
    n1 = a / math.sqrt(1 - e2 * sin_fp**2)
    d = x / (n1 * k0)

    lat = fp - (n1 * tan_fp / r1) * (
        d**2 / 2
        - (5 + 3 * t1 + 10 * c1 - 4 * c1**2 - 9 * ep2) * d**4 / 24
        + (61 + 90 * t1 + 298 * c1 + 45 * t1**2 - 252 * ep2 - 3 * c1**2) * d**6 / 720
    )
    lon = lon0 + (
        d
        - (1 + 2 * t1 + c1) * d**3 / 6
        + (5 - 2 * c1 + 28 * t1 - 3 * c1**2 + 8 * ep2 + 24 * t1**2) * d**5 / 120
    ) / cos_fp

    return math.degrees(lon), math.degrees(lat)


def load_grid_table(csv_path: str | Path = GRID_CSV_PATH) -> pd.DataFrame:
    csv_path = Path(csv_path)

    grid_df = pd.read_csv(csv_path)
    required_cols = {"id", "X", "Y", "left", "top", "right", "bottom"}
    missing_cols = required_cols - set(grid_df.columns)
    if missing_cols:
        raise ValueError(f"Missing required columns: {sorted(missing_cols)}")

    grid_df = grid_df.copy()
    grid_df["grid_id"] = grid_df["id"].astype(int)
    return grid_df


def get_hex_grid_geometry(grid_df: pd.DataFrame, grid_id: int) -> dict:
    row = grid_df.loc[grid_df["grid_id"] == grid_id]
    if row.empty:
        raise ValueError(f"grid_id={grid_id} was not found")

    row = row.iloc[0]
    left = float(row["left"])
    right = float(row["right"])
    top = float(row["top"])
    bottom = float(row["bottom"])
    cx = (left + right) / 2
    cy = (top + bottom) / 2
    w = right - left
    h = top - bottom

    vertices_3826 = [
        (cx - w / 2, cy),
        (cx - w / 4, cy + h / 2),
        (cx + w / 4, cy + h / 2),
        (cx + w / 2, cy),
        (cx + w / 4, cy - h / 2),
        (cx - w / 4, cy - h / 2),
    ]
    vertices_lonlat = [twd97_to_lonlat(x, y) for x, y in vertices_3826]

    min_lon = min(p[0] for p in vertices_lonlat)
    max_lon = max(p[0] for p in vertices_lonlat)
    min_lat = min(p[1] for p in vertices_lonlat)
    max_lat = max(p[1] for p in vertices_lonlat)

    return {
        "grid_id": int(row["grid_id"]),
        "center_lng": float(row["X"]),
        "center_lat": float(row["Y"]),
        "center_x_3826": cx,
        "center_y_3826": cy,
        "vertices_3826": vertices_3826,
        "vertices_lonlat": vertices_lonlat,
        "west": min_lon,
        "south": min_lat,
        "east": max_lon,
        "north": max_lat,
        "bbox_query": f"{min_lon},{min_lat},{max_lon},{max_lat}",
    }


# Backward-compatible alias used by the Mapillary cell.
def get_grid_coordinates(grid_df: pd.DataFrame, grid_id: int) -> dict:
    return get_hex_grid_geometry(grid_df, grid_id)


grid_df = load_grid_table()
display(grid_df.head())

# Change this to the grid you want to inspect.
# TARGET_GRID_ID = int(grid_df.iloc[0]["grid_id"])
# grid_coords = get_hex_grid_geometry(grid_df, TARGET_GRID_ID)
# grid_coords


,X,Y,id,left,top,right,bottom,row_index,col_index,fid,...,COUNTYNAME,TOWNNAME,VILLNAME,VILLENG,COUNTYID,COUNTYCODE,TOWNID,TOWNCODE,NOTE,grid_id
0,120.323629,23.130739,163914,180152.131280,2.559421e+06,181306.831819,2.558421e+06,371,291,1.0,...,臺南市,善化區,嘉南里,Jianan Vil.,D,67000.0,D27,67000190.0,NaN,163914
1,120.332062,23.135290,164476,181018.156684,2.559921e+06,182172.857222,2.558921e+06,371,292,1.0,...,臺南市,善化區,嘉南里,Jianan Vil.,D,67000.0,D27,67000190.0,NaN,164476
2,120.332107,23.126261,164477,181018.156684,2.558921e+06,182172.857222,2.557921e+06,372,292,1.0,...,臺南市,善化區,嘉南里,Jianan Vil.,D,67000.0,D27,67000190.0,NaN,164477
3,120.332196,23.108201,164479,181018.156684,2.556921e+06,182172.857222,2.555921e+06,374,292,1.0,...,臺南市,善化區,嘉南里,Jianan Vil.,D,67000.0,D27,67000190.0,NaN,164479
4,120.340496,23.139841,165037,181884.182088,2.560421e+06,183038.882626,2.559421e+06,370,293,2.0,...,臺南市,善化區,嘉北里,Jiabei Vil.,D,67000.0,D27,67000190.0,NaN,165037


In [8]:
import json
import random
from pathlib import Path

import pandas as pd
import requests

try:
    import mapillary.interface as mly
except ImportError as exc:
    raise ImportError(
        "Please install dependencies first: pip install mapillary requests pandas"
    ) from exc


MAPILLARY_ACCESS_TOKEN = "MLY|26199948926374186|6135d688b588deeabb91aedf98e975bc"
OUTPUT_DIR = Path("mapillary_downloads")


def _to_dict(payload):
    if isinstance(payload, str):
        return json.loads(payload)
    return payload


def _image_properties(image_meta: dict) -> dict:
    if isinstance(image_meta, dict) and isinstance(image_meta.get("properties"), dict):
        return image_meta["properties"]
    return image_meta if isinstance(image_meta, dict) else {}


def _image_url_from_metadata(image_meta: dict) -> str | None:
    props = _image_properties(image_meta)
    for key in ["thumb_original_url", "thumb_2048_url", "thumb_1024_url", "thumb_256_url"]:
        if props.get(key):
            return props[key]
    return None


def _coordinates_from_geometry(geometry: dict | None) -> tuple[float, float] | None:
    if not isinstance(geometry, dict):
        return None
    coordinates = geometry.get("coordinates")
    if not isinstance(coordinates, list) or len(coordinates) < 2:
        return None
    lon, lat = coordinates[:2]
    if lon is None or lat is None:
        return None
    return float(lon), float(lat)


def _feature_coordinates(feature: dict) -> tuple[float, float] | None:
    props = feature.get("properties", {}) if isinstance(feature, dict) else {}
    return (
        _coordinates_from_geometry(feature.get("geometry"))
        or _coordinates_from_geometry(props.get("computed_geometry"))
        or _coordinates_from_geometry(props.get("geometry"))
    )


def _image_coordinates(image_meta: dict) -> tuple[float, float] | None:
    props = _image_properties(image_meta)
    return (
        _coordinates_from_geometry(props.get("computed_geometry"))
        or _coordinates_from_geometry(image_meta.get("geometry"))
        or _coordinates_from_geometry(props.get("geometry"))
    )


def _point_in_polygon(lon: float, lat: float, polygon: list[tuple[float, float]]) -> bool:
    inside = False
    j = len(polygon) - 1
    for i, (xi, yi) in enumerate(polygon):
        xj, yj = polygon[j]
        on_lat_band = (yi > lat) != (yj > lat)
        if on_lat_band:
            intersect_lon = (xj - xi) * (lat - yi) / (yj - yi) + xi
            if lon <= intersect_lon:
                inside = not inside
        j = i
    return inside


def search_mapillary_images_in_grid(
    grid_df: pd.DataFrame,
    grid_id: int,
    sample_size: int,
    access_token: str,
    image_type: str = "flat",
    organization_id: str | None = None,
    min_captured_at: str | None = None,
    max_captured_at: str | None = None,
    random_seed: int | None = 42,
):
    if not access_token or access_token == "REPLACE_WITH_YOUR_MAPILLARY_ACCESS_TOKEN":
        raise ValueError("Please replace MAPILLARY_ACCESS_TOKEN with a valid token")

    hex_grid = get_hex_grid_geometry(grid_df, grid_id)
    bbox_query = {
        "west": hex_grid["west"],
        "south": hex_grid["south"],
        "east": hex_grid["east"],
        "north": hex_grid["north"],
    }
    hex_polygon = hex_grid["vertices_lonlat"]

    mly.set_access_token(access_token)

    filters = {}
    if image_type:
        filters["image_type"] = image_type
    if organization_id:
        filters["organization_id"] = organization_id
    if min_captured_at:
        filters["min_captured_at"] = min_captured_at
    if max_captured_at:
        filters["max_captured_at"] = max_captured_at

    raw_result = mly.images_in_bbox(bbox=bbox_query, **filters)
    result = _to_dict(raw_result)
    features = result.get("features", [])
    contained_features = []
    for feature in features:
        coordinates = _feature_coordinates(feature)
        if coordinates and _point_in_polygon(coordinates[0], coordinates[1], hex_polygon):
            contained_features.append(feature)

    if not contained_features:
        print(f"No Mapillary images were inside hex grid_id={grid_id}")
        return []

    rng = random.Random(random_seed)
    selected_features = rng.sample(contained_features, k=min(sample_size, len(contained_features)))
    print(
        f"Found {len(features)} bbox candidates, "
        f"kept {len(contained_features)} inside the hex, "
        f"and selected {len(selected_features)} images"
    )
    return selected_features


def download_mapillary_images(
    selected_features: list,
    access_token: str,
    output_dir: str | Path,
    grid_id: int,
    hex_polygon: list[tuple[float, float]] | None = None,
):
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    mly.set_access_token(access_token)

    if hex_polygon is None:
        hex_polygon = get_hex_grid_geometry(grid_df, grid_id)["vertices_lonlat"]

    records = []
    for idx, feature in enumerate(selected_features, start=1):
        props = feature.get("properties", {})
        image_id = str(props.get("id") or feature.get("id"))
        feature_coordinates = _feature_coordinates(feature)

        image_meta = _to_dict(
            mly.image_from_key(
                key=image_id,
                fields=[
                    "captured_at",
                    "thumb_original_url",
                    "thumb_2048_url",
                    "thumb_1024_url",
                    "thumb_256_url",
                    "computed_geometry",
                    "geometry",
                ],
            )
        )
        metadata_coordinates = _image_coordinates(image_meta)
        coordinates = feature_coordinates or metadata_coordinates
        if not coordinates:
            print(f"Skipped image_id={image_id} because no coordinate was returned")
            continue
        if not _point_in_polygon(coordinates[0], coordinates[1], hex_polygon):
            print(f"Skipped image_id={image_id} because its coordinate is outside grid_id={grid_id}")
            continue
        if metadata_coordinates and not _point_in_polygon(metadata_coordinates[0], metadata_coordinates[1], hex_polygon):
            print(f"image_id={image_id} metadata coordinate is outside; using the bbox-filtered feature coordinate")

        image_url = _image_url_from_metadata(image_meta)
        if not image_url:
            try:
                image_url = mly.image_thumbnail(image_id=image_id, resolution=2048)
            except Exception:
                image_url = None

        if not image_url:
            print(f"Skipped image_id={image_id} because no thumbnail URL was returned")
            continue

        response = requests.get(image_url, timeout=60)
        response.raise_for_status()

        save_path = output_dir / f"grid_{grid_id}_{idx:03d}_{image_id}.jpg"
        save_path.write_bytes(response.content)

        props = _image_properties(image_meta)
        records.append(
            {
                "grid_id": grid_id,
                "image_id": image_id,
                "captured_at": props.get("captured_at") or image_meta.get("captured_at"),
                "longitude": coordinates[0],
                "latitude": coordinates[1],
                "metadata_longitude": metadata_coordinates[0] if metadata_coordinates else None,
                "metadata_latitude": metadata_coordinates[1] if metadata_coordinates else None,
                "download_url": image_url,
                "save_path": str(save_path),
            }
        )
        print(f"[{idx}/{len(selected_features)}] Downloaded {save_path.name}")

    return pd.DataFrame(records)


# Change these two values before running.
TARGET_GRID_ID = int(grid_df.iloc[800]["grid_id"])
TARGET_IMAGE_COUNT = 10
hex_grid = get_hex_grid_geometry(grid_df, TARGET_GRID_ID)

selected_features = search_mapillary_images_in_grid(
    grid_df=grid_df,
    grid_id=TARGET_GRID_ID,
    sample_size=TARGET_IMAGE_COUNT,
    access_token=MAPILLARY_ACCESS_TOKEN,
    image_type="flat",  # You can change this to "pano" or "all".
    random_seed=42,
)

download_df = download_mapillary_images(
    selected_features=selected_features,
    access_token=MAPILLARY_ACCESS_TOKEN,
    output_dir=OUTPUT_DIR / f"grid_{TARGET_GRID_ID}",
    grid_id=TARGET_GRID_ID,
    hex_polygon=hex_grid["vertices_lonlat"],
)

download_df


Requesting GET to https://tiles.mapillary.com/maps/vtp/mly1_public/2/14/13664/7111/?access_token=MLY%7C26199948926374186%7C6135d688b588deeabb91aedf98e975bc
Response 200 OK received in 492ms
Requesting GET to https://tiles.mapillary.com/maps/vtp/mly1_public/2/14/13665/7111/?access_token=MLY%7C26199948926374186%7C6135d688b588deeabb91aedf98e975bc
Response 200 OK received in 324ms
Found 2043 bbox candidates, kept 863 inside the hex, and selected 10 images
Requesting GET to https://graph.mapillary.com/1397094258455999/?fields=captured_at,thumb_original_url,thumb_2048_url,thumb_1024_url,thumb_256_url,computed_geometry,geometry&access_token=MLY%7C26199948926374186%7C6135d688b588deeabb91aedf98e975bc
Response 200 OK received in 451ms
Requesting GET to https://graph.mapillary.com/1397094258455999/?fields=thumb_2048_url,geometry&access_token=MLY%7C26199948926374186%7C6135d688b588deeabb91aedf98e975bc
Response 200 OK received in 408ms
[1/10] Downloaded grid_159423_001_1397094258455999.jpg
Requestin

,grid_id,image_id,captured_at,longitude,latitude,metadata_longitude,metadata_latitude,download_url,save_path
0,159423,1397094258455999,None,120.258831,23.082865,None,None,https://scontent.fkhh5-1.fna.fbcdn.net/m1/v/t6...,mapillary_downloads\grid_159423\grid_159423_00...
1,159423,1214328350679986,None,120.254336,23.089375,None,None,https://scontent.fkhh5-1.fna.fbcdn.net/m1/v/t6...,mapillary_downloads\grid_159423\grid_159423_00...
2,159423,625221073998107,None,120.251294,23.086093,None,None,https://scontent.fkhh5-1.fna.fbcdn.net/m1/v/t6...,mapillary_downloads\grid_159423\grid_159423_00...
3,159423,1772422584162228,None,120.257780,23.081232,None,None,https://scontent.fkhh5-1.fna.fbcdn.net/m1/v/t6...,mapillary_downloads\grid_159423\grid_159423_00...
4,159423,169455851661965,None,120.251498,23.085367,None,None,https://scontent.fkhh5-1.fna.fbcdn.net/m1/v/t6...,mapillary_downloads\grid_159423\grid_159423_00...
5,159423,1309019956215822,None,120.251445,23.084094,None,None,https://scontent.fkhh5-1.fna.fbcdn.net/m1/v/t6...,mapillary_downloads\grid_159423\grid_159423_00...
6,159423,1147674756340646,None,120.251874,23.083364,None,None,https://scontent.fkhh5-1.fna.fbcdn.net/m1/v/t6...,mapillary_downloads\grid_159423\grid_159423_00...
7,159423,704184982643307,None,120.251568,23.084030,None,None,https://scontent.fkhh5-1.fna.fbcdn.net/m1/v/t6...,mapillary_downloads\grid_159423\grid_159423_00...
8,159423,846596001418964,None,120.258381,23.081676,None,None,https://scontent.fkhh5-1.fna.fbcdn.net/m1/v/t6...,mapillary_downloads\grid_159423\grid_159423_00...
9,159423,1184664812839425,None,120.251482,23.086216,None,None,https://scontent.fkhh5-1.fna.fbcdn.net/m1/v/t6...,mapillary_downloads\grid_159423\grid_159423_01...
